# Stage 2: Representative Set Selection and Crop Preparation

## Overview and Purpose
In computer vision workflows for character clustering, extracting feature embeddings from whole, uncropped images is sub-optimal. Full images often feature dominant backgrounds, scenery, multiple figures, or distracting visual elements that overwhelm character identity. Character visual identity is predominantly concentrated in facial structure, hair color and geometry, eye shapes, and upper-body costume details.

This notebook executes **Stage 2** of the pipeline:
1. **Representative Set Selection**: Excludes non-representative exact duplicates while optionally retaining perceptual duplicates for diversity.
2. **Anime Region Detection**: Runs specialized YOLO-based anime head and person detectors from `dghs-imgutils`.
3. **Region Priority Hierarchy**: Selects the optimal crop region following the deterministic hierarchy:
   $$\text{head crop} \longrightarrow \text{person crop} \longrightarrow \text{full image fallback}$$
4. **Bounding Box Expansion & Clamping**: Expands bounding boxes by a generous contextual padding ratio (default: 25%) to preserve hairstyles, accessories, ribbons, and hats.
5. **Quality Review Flagging**: Identifies problematic crops (no detection, multiple heads, tiny detections) and flags them for manual inspection.
6. **Visual Inspection**: Generates HTML/JPEG contact sheets for visual auditing.

## 1. Imports and Environment Setup
Import required packages and modular preprocessing components.

In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import Image as IPImage, display

from anime_character_organizer import (
    CropConfig,
    run_crop_preparation,
)
from anime_character_organizer.preprocessing.geometry import clamp_box, expand_box
from anime_character_organizer.preprocessing.detection import choose_best_detection
from anime_character_organizer.utils.runs import find_latest_run

## 2. Configuration & Parameter Tuning
Configure detection thresholds, padding parameters, and output dimensions.

In [ ]:
PROJECT_DIR = Path("./anime_character_pipeline").expanduser().resolve()

# Auto-discover the latest valid Stage 1 audit run
PREVIOUS_AUDIT_RUN = None

# Set to False (recommended) so diverse poses/angles of the same character are NOT discarded
USE_PERCEPTUAL_REPRESENTATIVES = False

# Detection confidence thresholds
HEAD_CONF_THRESHOLD = 0.35
PERSON_CONF_THRESHOLD = 0.30

# Contextual padding: 25% expands the head box to include full hair, ahoge, and accessories
HEAD_PADDING_RATIO = 0.25
PERSON_PADDING_RATIO = 0.10

# Output crop specifications
CROP_OUTPUT_SIZE = (512, 512)
CROP_FORMAT = "JPEG"
CROP_QUALITY = 95
MIN_CROP_SIDE = 64

print("Crop Preparation Parameters:")
print(f"  Head Confidence: {HEAD_CONF_THRESHOLD}, Padding: {HEAD_PADDING_RATIO}")
print(f"  Person Confidence: {PERSON_CONF_THRESHOLD}, Padding: {PERSON_PADDING_RATIO}")
print(f"  Output Size: {CROP_OUTPUT_SIZE}, Format: {CROP_FORMAT}")

## 3. Crop Preparation Workflow Execution
Execute the crop preparation workflow. The function auto-discovers the latest Stage 1 audit run, extracts crops in parallel, and saves them under `runs/02_crop_preparation_<timestamp>/`.

In [ ]:
results = run_crop_preparation(
    project_dir=PROJECT_DIR,
    previous_run_dir=PREVIOUS_AUDIT_RUN,
    use_perceptual_representatives=USE_PERCEPTUAL_REPRESENTATIVES,
    head_conf_threshold=HEAD_CONF_THRESHOLD,
    person_conf_threshold=PERSON_CONF_THRESHOLD,
    head_padding_ratio=HEAD_PADDING_RATIO,
    person_padding_ratio=PERSON_PADDING_RATIO,
    crop_output_size=CROP_OUTPUT_SIZE,
    crop_format=CROP_FORMAT,
    crop_quality=CROP_QUALITY,
    min_crop_side=MIN_CROP_SIDE,
    show_progress=True,
)

print(f"Crop preparation completed.")
print(f"Run directory: {results['run_dir']}")

## 4. Execution Summary & Crop Quality Verification
Inspect the distribution of extracted region types and review flags.

In [ ]:
summary = results["summary"]
summary_df = pd.DataFrame([{
    "Valid Images Loaded": summary["valid_images_loaded"],
    "Selected for Detection": summary["selected_for_detection"],
    "Successful Crops": summary["successful_crops"],
    "Crop Errors": summary["crop_errors"],
    "Review Required": summary["review_required"],
}])
display(summary_df)

## 5. Region Distribution Analysis
Inspect how many images utilized head detections, person detections, or full-image fallbacks.

In [ ]:
region_counts_df = results["region_counts"]
print("Extracted Region Type Counts:")
display(region_counts_df)

## 6. Review Reason Breakdown
Images flagged with `needs_review=True` are cataloged with specific review reasons to guide manual inspection.

In [ ]:
review_counts_df = results["review_counts"]
print("Review Reasons Breakdown:")
display(review_counts_df)

## 7. Inspection of Review Items & Contact Sheets
Inspect sample rows of crops requiring manual review, and check generated diagnostic contact sheets.

In [ ]:
review_df = results["review_df"]
review_needed_df = review_df[review_df["needs_review"]].copy()

if review_needed_df.empty:
    print("Success: Zero crops required manual review flags.")
else:
    print(f"Crops Requiring Review ({len(review_needed_df)} items):")
    display(review_needed_df[[
        "crop_id", "selected_region_type", "detected_heads_count",
        "detected_persons_count", "review_reason"
    ]].head(25))

contact_sheets_df = results["contact_sheets_df"]
if not contact_sheets_df.empty:
    print("Generated Contact Sheets:")
    display(contact_sheets_df)

## 8. Assumptions, Limitations, and Next Steps

### Assumptions & Limitations
- **Multi-Character Scenes**: When multiple heads are detected (`is_multi_head=True`), the detection heuristic selects the most prominent head based on score and bounding box area. Such images are automatically flagged with `multiple_heads_detected` for auditing.
- **Art Style Variation**: Extreme stylizations (chibi, silhouette, non-human faces) may fall back to full image mode, which preserves context rather than failing.

### Next Steps
The crop manifest `crop_manifest_with_review_flags.csv` is passed to **Stage 3 (`03_ccip_embedding_extraction.ipynb`)** for CCIP character similarity embedding extraction.